In [ ]:
main_path = '/content/drive/MyDrive/Colab Notebooks/'


In [69]:
import pandas as pd
import os
main_path = '/content/drive/MyDrive/Colab Notebooks/'

def add_feature(df):
    # df['Year'] = df.index.year
    df['Month'] = df.index.month
    df['Day'] = df.index.day
    df['Dayofweek'] = df.index.dayofweek
    df['Weekend'] = df.index.dayofweek >= 5
    df['Hour'] = df.index.hour
    df['Daylight'] = df['Hour'].apply(lambda x: 1 if 7 <= x <= 19 else 0)
    return df

def Alexandra(id_number=1):
    # Get the path to the folder containing the CSV files
    sub_path = 'Data/Alexandra/'
    folder_path = main_path + sub_path

    # List all files in the folder
    files = os.listdir(folder_path)

    # Filter the files to only include CSV files
    csv_files = [file for file in files if file.endswith('.csv')]

    # Define the data
    df = pd.read_csv(main_path + sub_path + csv_files[id_number])
    # Let's convert 'DateTime' column to datetime type if it's not already
    df.rename(columns={'energy': 'VAL'}, inplace=True)

    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df['DateTime'] += pd.to_timedelta(df['hour'], unit='h')
    df= df[['DateTime','VAL']]

    df.index = df['DateTime']
    df.drop(columns='DateTime', inplace=True)
    df = df[~df.index.duplicated(keep='first')]
    df = add_feature(df)

    return df

def Muhammad(id_number=1):
    # Get the path to the folder containing the CSV files
    sub_path = 'Data/Muhammad/'
    folder_path = main_path + sub_path

    # List all files in the folder
    files = os.listdir(folder_path)

    # Filter the files to only include CSV files
    csv_files = [file for file in files if file.endswith('.csv')]

    # Define the data
    df = pd.read_csv(main_path + sub_path + csv_files[id_number])

    # Let's convert 'DateTime' column to datetime type if it's not already
    df.rename(columns={'PJME_MW': 'VAL', 'Datetime':'DateTime'}, inplace=True)

    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df= df[['DateTime','VAL']]

    df.index = df['DateTime']
    df.drop(columns='DateTime', inplace=True)
    df = df[~df.index.duplicated(keep='first')]

    df = add_feature(df)

    return df

def Kaggle1(id_number=1):
    # Get the path to the folder containing the CSV files
    sub_path = 'Data/Kaggle1/'
    folder_path = main_path + sub_path

    # List all files in the folder
    files = os.listdir(folder_path)

    # Filter the files to only include CSV files
    csv_files = [file for file in files if file.endswith('.csv')]

    # Define the data
    df = pd.read_csv(main_path + sub_path + csv_files[id_number])

    # Let's convert 'DateTime' column to datetime type if it's not already
    df.rename(columns={'Datetime':'DateTime'}, inplace=True)
    df.rename(columns={df.columns[1]: 'VAL'}, inplace=True)

    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df= df[['DateTime','VAL']]

    df.index = df['DateTime']
    df.drop(columns='DateTime', inplace=True)
    df = df[~df.index.duplicated(keep='first')]

    df = add_feature(df)

    return df

In [66]:
def load_and_preprocess_data_with_sequences(df, target_col_name="VAL", scale_type='standardize', train_ratio=0.75, val_ratio=0.15, test_ratio=0.10, input_steps=48, output_steps=24):
    # Drop NA values or consider imputation
    df.dropna(inplace=True)

    # Extract features and target
    target = df[[target_col_name]].values
    features = df.drop(columns=[target_col_name]).values  # Make sure to drop the target column from features

    # Determine scaler type
    if scale_type == 'normalize':
        scaler_features = MinMaxScaler(feature_range=(-1, 1))
        scaler_target = MinMaxScaler(feature_range=(-1, 1))
    elif scale_type == 'standardize':
        scaler_features = StandardScaler()
        scaler_target = StandardScaler()
    else:
        raise ValueError("scale_type should be either 'normalize' or 'standardize'")

    # Fit scaler only on the training data
    total_length = len(df)
    train_end = int(total_length * train_ratio)
    scaler_features.fit(features[:train_end])
    scaler_target.fit(target[:train_end])

    # Transform all feature and target data
    features_scaled = scaler_features.transform(features)
    target_scaled = scaler_target.transform(target)

    # Combine scaled features with scaled target for proper sequence creation
    all_data_scaled = np.concatenate((features_scaled, target_scaled), axis=1)

    # Split data into train, validation, and test
    val_end = train_end + int(total_length * val_ratio)

    X_train, y_train = create_sequences(all_data_scaled[:train_end, :-1], all_data_scaled[:train_end, -1], input_steps, output_steps)
    X_val, y_val = create_sequences(all_data_scaled[train_end:val_end, :-1], all_data_scaled[train_end:val_end, -1], input_steps, output_steps)
    X_test, y_test = create_sequences(all_data_scaled[val_end:, :-1], all_data_scaled[val_end:, -1], input_steps, output_steps)

    return X_train, y_train, X_val, y_val, X_test, y_test, input_steps, output_steps


In [67]:
# Example usage:
df = Alexandra(id_number=2)
X_train, y_train, X_val, y_val, X_test, y_test, input_steps, output_steps = load_and_preprocess_data_with_sequences(df, input_steps=48, output_steps=24)

In [68]:
y_train

array([[-0.93742972, -1.04754502, -1.03331773, ...,  0.49426399,
         0.02788187, -0.3576191 ],
       [-1.04754502, -1.03331773, -1.01909045, ...,  0.02788187,
        -0.3576191 , -0.60357574],
       [-1.03331773, -1.01909045, -0.91267815, ..., -0.3576191 ,
        -0.60357574, -0.59889828],
       ...,
       [-0.43421257, -0.09743518,  0.64316326, ..., -0.63924141,
        -0.66730619, -0.69342203],
       [-0.09743518,  0.64316326,  0.14131037, ..., -0.66730619,
        -0.69342203, -0.62189581],
       [ 0.64316326,  0.14131037,  0.56618   , ..., -0.69342203,
        -0.62189581, -0.48469021]])